# DwC-A Cloud Geospatial demo

This notebook demonstrates the repository workflow in Google Colab: install the converter, download a GBIF Darwin Core Archive, inspect the archive structure, convert occurrence records into cloud-friendly geospatial outputs, validate the generated bundle and download the results.

Source repository: [ABiatov/dwca-cloud-geospatial](https://github.com/ABiatov/dwca-cloud-geospatial).

The workflow is file-based. It does not require a database, backend service or permanent cloud deployment; the generated bundle can be copied to static hosting.

## Dataset used in this demo

This demo uses the GBIF occurrence download with key `0038004-260519110011954`.

Citation to preserve with any reused outputs:

> GBIF.org (4 June 2026) GBIF Occurrence Download https://doi.org/10.15468/dl.3xbk5b

License: `CC_BY_NC_4_0`.

The converter records the download key, DOI, citation and license in the generated bundle metadata when they are passed during conversion.

## 1. Install the converter in Colab

Clone the repository and install the package with the FlatGeobuf and validation extras. This installs the CLI used below and the optional writer/validator dependencies needed for the full demo bundle.

In [ ]:
from google.colab import output
from google.colab import files

!git clone https://github.com/ABiatov/dwca-cloud-geospatial.git
%cd /content/dwca-cloud-geospatial
%pip install -e ".[dev,flatgeobuf,validation]"
output.clear()

## 2. Download the source Darwin Core Archive

The archive is downloaded from the GBIF occurrence download API and saved outside the cloned repository at `/content/0038004-260519110011954.zip`.

In [ ]:
GBIF_download_key = "0038004-260519110011954"

!curl -L \
  "https://api.gbif.org/v1/occurrence/download/request/{GBIF_download_key}.zip" \
  -o ../{GBIF_download_key}.zip

## 3. Inspect the archive

`inspect` reads the DwC-A `meta.xml` declaration and reports whether the archive contains an Occurrence core and coordinate fields required for geospatial conversion.

In [ ]:
!python -m dwca_cloud_geospatial inspect ../{GBIF_download_key}.zip

## 4. Convert to a static geospatial bundle

The Python API writes the same portable output bundle as the CLI. This demo generates both FlatGeobuf for the browser map layer and GeoParquet for analytical workflows. `GbifDownloadOptions(enrich=True)` opts in to read-only GBIF metadata lookup so the generated metadata preserves the source dataset attribution. The code cell adds the cloned repository's `src/` directory to `sys.path` so imports work reliably in notebook kernels.

In [ ]:
from pathlib import Path
import sys

repo_root = Path("/content/dwca-cloud-geospatial")
repo_src = repo_root / "src"
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from dwca_cloud_geospatial.conversion import ConversionOptions, convert_dwca_archive
from dwca_cloud_geospatial.gbif import GbifDownloadOptions
from dwca_cloud_geospatial.geoparquet import GeoParquetWriterOptions

archive_path = Path(f"../{GBIF_download_key}.zip")
output_path = Path("../output")

result = convert_dwca_archive(
    archive_path,
    output_path,
    options=ConversionOptions(
        output_formats=("flatgeobuf", "geoparquet"),
        overwrite=True,
        gbif=GbifDownloadOptions(enrich=True),
        geoparquet=GeoParquetWriterOptions(large_output_mode=True),
        chunk_size=10000,
    ),
)
print(f"Bundle successfully created at: {result.output_directory}")

## 5. Validate the output bundle

`validate` checks the generated bundle structure, manifest, metadata and geospatial outputs. Required validation errors should be fixed before sharing the bundle.

In [ ]:
!python -m dwca_cloud_geospatial validate ../output

## 6. Run analytical SQL against GeoParquet

The generated GeoParquet file can be queried directly with DuckDB. Because the converter includes a built-in GeoParquet writer, you can quickly build SQL pipelines without configuring a PostGIS database.

This practical example extracts threatened or near-threatened species recorded since 1995, using IUCN Red List categories for Critically Endangered (`CR`), Endangered (`EN`), Vulnerable (`VU`) and Near Threatened (`NT`).

The same pattern can query cloud-hosted GeoParquet files by replacing the local path with a URL, for example `https://abiatov.github.io/dwca-cloud-geospatial/demo/output/data/occurrences.parquet`.

In [ ]:
import duckdb

# Initialize the connection and load the spatial module
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# Perform an analytical query directly to a local or cloud file 
query = """
SELECT 
    occurrence_id, 
    scientific_name, 
    kingdom,
    event_year, 
    iucn_red_list_category, 
    decimal_longitude, 
    decimal_latitude 
FROM read_parquet('../output/data/occurrences.parquet')
WHERE event_year >= 1995 
  AND iucn_red_list_category IN ('CR', 'EN', 'VU', 'NT')
LIMIT 10;
"""

print(con.execute(query).df())

## 7. Download individual geospatial artifacts

Use these cells when you want separate copies of the generated FlatGeobuf, retained GeoPackage staging artifact and GeoParquet output.

In [ ]:
files.download('/content/output/data/occurrences.fgb')

In [ ]:
files.download('/content/output/data/occurrences.gpkg')

In [ ]:
files.download('/content/output/data/occurrences.parquet')

## 8. Download the complete static bundle

The complete bundle includes the viewer files, manifest, metadata, reports and generated geospatial data. Preserve the citation above when redistributing derived outputs.

In [ ]:
!zip -r /content/output_{GBIF_download_key}.zip /content/output


In [ ]:
files.download(f'/content/output_{GBIF_download_key}.zip')